In [30]:
import pandas as pd

df_labels = pd.read_csv("dataset_labels.csv")
df_labels

,file_path,filename,is_doc_id,id_doc_type,doc_type
0,dataset/false_doc,12031810-405818722.jpg,False,NaN,NaN
1,dataset/false_doc,facture-d-avoir-exemple-744x1024-2687424784.jpg,False,NaN,NaN
2,dataset/false_id,1-18900169.jpg,False,NaN,NaN
3,dataset/false_id,1-842369233.jpg,False,NaN,NaN
4,dataset/false_id,Document-d-identite-2.jpg,False,NaN,NaN
5,dataset/false_id,ID_passport_USA-2021-497107151.jpg,False,NaN,NaN
6,dataset/false_id,IMG_6211_LegalPhoto-1697863245.jpg,False,NaN,NaN
7,dataset/false_id,carte-d-identite-ce-qui-change-en-bretagne_317...,False,NaN,NaN
8,dataset/false_id,csm_CA_RPC_2021_1920_4c0a2d9287-515808575.jpg,False,NaN,NaN
9,dataset/false_id,les-faux-papiers-ne-sont-pas-toujours-facileme...,False,NaN,NaN


In [31]:
df_results = pd.read_csv("results.csv")
df_results

,file_path,filename,mistral_is_doc_id,mistral_id_doc_type,mistral_doc_type,azure_is_doc_id,azure_id_doc_type,azure_doc_type,mistral_vision_is_doc_id,mistral_vision_id_doc_type,mistral_vision_doc_type,azure_vision_is_doc_id,azure_vision_id_doc_type,azure_vision_doc_type,aligned
0,dataset/false_doc,12031810-405818722.jpg,True,proof_of_residency,letter,NaN,NaN,NaN,True,proof_of_residency,official_request_for_vehicle_registration_docu...,False,not_identity_doc,official request letter,False
1,dataset/false_doc,facture-d-avoir-exemple-744x1024-2687424784.jpg,False,not_identity_doc,credit_note,NaN,NaN,NaN,False,not_identity_doc,credit_note,False,not_identity_doc,credit note,False
2,dataset/false_id,1-18900169.jpg,True,id card,identity card,NaN,NaN,NaN,True,id_card,"[{'type': 'student_id', 'issuer': 'Université ...",True,id card,"student_id_card, voter_id_card",False
3,dataset/false_id,1-842369233.jpg,True,proof_of_residency,residence_permit,NaN,NaN,NaN,True,proof_of_residency,Titre de Séjour (French Residence Permit),True,proof_of_residency,titre de séjour,False
4,dataset/false_id,Document-d-identite-2.jpg,True,passport,passport,NaN,NaN,NaN,True,passport,travel_document,True,passport,passport,False
5,dataset/false_id,ID_passport_USA-2021-497107151.jpg,True,passport,passport,NaN,NaN,NaN,True,passport,government_issued_travel_document,True,passport,passport,False
6,dataset/false_id,IMG_6211_LegalPhoto-1697863245.jpg,True,id card,identity photo,NaN,NaN,NaN,True,id card,Legal identity photo printout conforming to IS...,True,not_identity_doc,passport photo sheet,False
7,dataset/false_id,carte-d-identite-ce-qui-change-en-bretagne_317...,True,id card,identity document,NaN,NaN,NaN,True,id_card,Carte Nationale d’Identité (French National Id...,True,id card,French national identity card,False
8,dataset/false_id,csm_CA_RPC_2021_1920_4c0a2d9287-515808575.jpg,True,proof_of_residency,permanent_resident_card,NaN,NaN,NaN,True,id card,Permanent Resident Card,True,proof_of_residency,permanent resident card,False
9,dataset/false_id,les-faux-papiers-ne-sont-pas-toujours-facileme...,True,id card,identity card,NaN,NaN,NaN,True,id_card,{'primary': ['Italian Identity Card (Carta d’I...,True,passport,"passport, id card, driving license",False


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve

MODELS = {
    "Mistral OCR":    "mistral_is_doc_id",
    "Azure CU":       "azure_is_doc_id",
    "Mistral Vision": "mistral_vision_is_doc_id",
    "Azure Vision":   "azure_vision_is_doc_id",
}
BOOL_MAP = {True: 1, False: 0, "True": 1, "False": 0}

# Merge ground truth with model results on filename
merged = df_labels[["filename", "is_doc_id"]].merge(df_results, on="filename", suffixes=("_gt", ""))
merged = merged.dropna(subset=["is_doc_id"])


def get_yt_ys(col):
    """Return (y_true, y_score) for a given model column, excluding missing predictions."""
    mask = merged[col].notna() & (merged[col] != "")
    y_t = merged.loc[mask, "is_doc_id"].astype(bool).astype(int)
    y_s = merged.loc[mask, col].map(BOOL_MAP).fillna(0).astype(int)
    return y_t, y_s


fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random (AUC = 0.50)")

for name, col in MODELS.items():
    y_t, y_s = get_yt_ys(col)
    if len(y_t) == 0 or y_t.nunique() < 2:
        print(f"[skip] {name}: not enough data (n={len(y_t)})")
        continue
    auc = roc_auc_score(y_t, y_s)
    fpr, tpr, _ = roc_curve(y_t, y_s)
    ax.plot(fpr, tpr, marker="o", label=f"{name} (AUC = {auc:.2f}, n={len(y_t)})")

ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC — is_doc_id classification")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, precision_score, recall_score

rows = []
for name, col in MODELS.items():
    y_t, y_s = get_yt_ys(col)
    if len(y_t) == 0 or y_t.nunique() < 2:
        continue
    rows.append({
        "model":     name,
        "n":         len(y_t),
        "accuracy":  accuracy_score(y_t, y_s),
        "precision": precision_score(y_t, y_s, zero_division=0),
        "recall":    recall_score(y_t, y_s, zero_division=0),
        "f1":        f1_score(y_t, y_s, zero_division=0),
        "mcc":       matthews_corrcoef(y_t, y_s),
        "auc":       roc_auc_score(y_t, y_s),
    })

(pd.DataFrame(rows)
   .set_index("model")
   .style
   .format("{:.2f}", subset=["accuracy", "precision", "recall", "f1", "mcc", "auc"])
   .background_gradient(cmap="RdYlGn", subset=["accuracy", "precision", "recall", "f1", "mcc", "auc"])
)

## Conclusion — Performance des modèles sur `is_doc_id`

L'analyse porte sur **40 fichiers** labellisés (19 documents d'identité, 21 non-identité), évalués sur trois modèles disposant de résultats complets (Azure CU n'a produit aucune prédiction et est exclu).

### Résultats

| Modèle | Accuracy | Precision | Recall | F1 | MCC | AUC |
|---|---|---|---|---|---|---|
| Mistral OCR | 0.78 | 0.71 | **1.00** | 0.83 | 0.60 | 0.75 |
| Mistral Vision | 0.75 | 0.69 | **1.00** | 0.81 | 0.55 | 0.72 |
| Azure Vision | 0.78 | 0.71 | **1.00** | 0.83 | 0.60 | 0.75 |

### Points clés

**Recall parfait (1.00) pour les trois modèles** — aucun document d'identité réel n'est manqué. C'est la propriété la plus critique dans un contexte de vérification documentaire : un faux négatif (document d'identité non détecté) serait une erreur grave.

**Precision modérée (~0.70)** — environ 30 % des documents classifiés comme "identité" ne le sont pas. Ces faux positifs proviennent principalement de la catégorie `false_id` (photos de documents, coupures de presse sur les cartes d'identité) que les modèles confondent avec de vrais documents.

**MCC entre 0.55 et 0.60** — performance correcte mais perfectible sur un dataset déséquilibré. Un MCC > 0.8 serait souhaitable en production.

**Mistral OCR et Azure Vision à égalité** (F1 = 0.83, MCC = 0.60) devant Mistral Vision (F1 = 0.81, MCC = 0.55). L'approche OCR n'apporte pas d'avantage décisif sur la vision pure.

### Limites de l'analyse

- **Dataset petit** (n=40 avec labels) — les métriques sont sensibles à quelques exemples.
- **Azure CU non évalué** — absence de résultats dans `results.csv`, à relancer.
- **Catégorie `false_id` difficile** — images de documents sur Internet (pas de vrais faux) : la tâche est plus ambiguë que des cas réels.

### Recommandation

Pour un usage production, **augmenter le dataset** (notamment les faux positifs) et **affiner le seuil de décision** sur `id_doc_type` pour réduire la precision sans dégrader le recall.